# Problema de la mochila : Algoritmo Genético
Hugo André Meza Fierros

A00841695

# Problema 1

Un armador tiene un carguero con capacidad de hasta 800 toneladas. El carguero transporta contenedores de diferentes pesos para una determinada ruta. En la ruta actual el carguero puede transportar algunos de los siguientes contenedores:
| Contenedor | $c_1$ | $c_2$ | $c_3$ | $c_4$ | $c_5$ | $c_6$ | $c_7$ | $c_8$ | $c_9$ | $c_{10}$ |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **Peso [ton]** | 61 | 58 | 92 | 50 | 108 | 83 | 93 | 101 | 54 | 50 |
| **Beneficio [\$]** | 1100 | 1147 | 1442 | 1591 | 1078 | 1385 | 1777 | 1196 | 1753 | 1371 |

<br>

| Contenedor | $c_{11}$ | $c_{12}$ | $c_{13}$ | $c_{14}$ | $c_{15}$ | $c_{16}$ | $c_{17}$ | $c_{18}$ | $c_{19}$ | $c_{20}$ |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **Peso [ton]** | 72 | 51 | 100 | 108 | 91 | 112 | 66 | 58 | 110 | 73 |
| **Beneficio [\$]** | 1517 | 1675 | 1193 | 1177 | 1365 | 1143 | 1314 | 1526 | 1470 | 1605 |

El analista de la empresa del armador desea determinar el envío (conjunto de contenedores) que maximiza el beneficio.

## Método exacto (CPLEX)

In [1]:
from docplex.mp.model import Model
T = 800
I = range(1, 21)
w = [61, 58, 92, 50, 108, 83, 93, 101, 54, 50, 72, 51, 100, 108, 91, 112, 66, 58, 110, 73]
c = [1100, 1147, 1442, 1591, 1078, 1385, 1777, 1196, 1753, 1371, 1517, 1675, 1193, 1177, 1365, 1143, 1314, 1526, 1470, 1605]

In [6]:
mdl = Model('Mochila')

x = mdl.binary_var_list(I, name='x') # variable de decisión

mdl.maximize(mdl.sum(c[i-1] * x[i-1] for i in I)) # Función objetivo

mdl.add_constraint(mdl.sum(w[i-1] * x[i-1] for i in I) <= T, ctname='r_capacidad') # Restricción de capacidad

docplex.mp.LinearConstraint[r_capacidad](61x_1+58x_2+92x_3+50x_4+108x_5+83x_6+93x_7+101x_8+54x_9+50x_10+72x_11+51x_12+100x_13+108x_14+91x_15+112x_16+66x_17+58x_18+110x_19+73x_20,LE,800)

In [8]:
solucion = mdl.solve()

if solucion:
    # 1. Valor de la función objetivo Z
    valor_objetivo = solucion.objective_value
    
    # Obtener el objeto de detalles del solver
    detalles = mdl.get_solve_details()
    
    # 2. Tiempo computacional y número de iteraciones
    tiempo_cpu = detalles.time
    num_iteraciones = detalles.nb_iterations

    # 3. Solución factible (Contenedores donde x_i = 1)
    # x[i].solution_value da el valor asignado a la variable en la solución
    seleccionados = [i for i in I if x[i-1].solution_value > 0.5]

    # 4. Capacidad ocupada (Suma de pesos de los contenedores seleccionados)
    capacidad_ocupada = sum(w[i - 1] for i in seleccionados)
    porcentaje_ocupacion = (capacidad_ocupada / T) * 100

    # Imprimir reporte completo
    print("=" * 45)
    print("       REPORTE DE MÉTRICAS DE EJECUCIÓN (CPLEX)")
    print("=" * 45)
    print(f"Función Objetivo (Z) : ${valor_objetivo:,.2f}")
    print(f"Solución Factible (S): Contenedores {seleccionados}")
    print(f"Capacidad Ocupada    : {capacidad_ocupada} / {T} ton ({porcentaje_ocupacion:.1f}%)")
    print(f"Tiempo Computacional : {tiempo_cpu:.6f} segundos")
    print(f"Número de Iteraciones: {num_iteraciones}")
    print("=" * 45)
else:
    print("No se encontró una solución factible.")

       REPORTE DE MÉTRICAS DE EJECUCIÓN (CPLEX)
Función Objetivo (Z) : $18,103.00
Solución Factible (S): Contenedores [2, 3, 4, 6, 7, 9, 10, 11, 12, 17, 18, 20]
Capacidad Ocupada    : 800 / 800 ton (100.0%)
Tiempo Computacional : 0.016000 segundos
Número de Iteraciones: 2


## Algoritmo genético

In [9]:
import numpy as np

c = np.array(c)
w = np.array(w)
T = 800 # Capacidad máxima de la mochila
n = len(c) # Número de contenedores (longitud)
num_vectores = 11 # Tamaño de la población

vectores_numpy = np.random.randint(0, 2, size=(num_vectores, n))
print(vectores_numpy)


[[1 0 1 0 1 1 1 0 1 0 1 0 1 1 1 0 1 1 0 1]
 [1 0 0 1 0 0 0 0 0 0 1 1 0 0 1 1 0 1 1 1]
 [0 1 1 1 1 0 1 1 1 0 0 1 0 0 1 0 0 1 0 1]
 [0 0 0 0 0 1 1 0 1 0 0 0 1 1 0 1 1 0 1 1]
 [0 0 1 0 0 0 1 0 0 0 0 1 0 1 1 0 1 1 1 0]
 [1 0 1 0 1 0 1 0 1 1 1 0 0 0 1 1 0 1 0 1]
 [1 0 1 0 1 0 0 1 1 0 1 1 0 1 0 0 0 1 1 1]
 [1 1 0 1 0 1 1 1 1 0 1 1 0 1 1 0 0 0 0 1]
 [1 0 0 0 0 1 0 1 0 0 1 1 1 1 0 1 0 0 0 1]
 [1 0 1 0 0 1 1 1 0 0 0 0 0 0 0 0 0 1 0 1]
 [0 1 0 0 0 0 1 0 1 1 0 1 1 1 0 1 1 1 0 1]]


In [14]:
def objective_function(c, w, T, vector):
    total_value = np.sum(c * vector)
    total_weight = np.sum(w * vector)
    return total_value, total_weight

def fitness_function(c, w, T, vector):
    total_value, total_weight = objective_function(c, w, T, vector)
    if total_weight > T:
        return total_value - (total_weight - T) * 100  # Penalización por exceder la capacidad
    else:
        return total_value

def ruleta(fitness_values):
    total_fitness = np.sum(fitness_values)
    probabilities = fitness_values / total_fitness
    cumulative_probabilities = np.cumsum(probabilities)
    r = np.random.rand()
    for i, cp in enumerate(cumulative_probabilities):
        if r < cp:
            return i

In [15]:
soluciones = []

for vector in vectores_numpy:
    total_value, total_weight = objective_function(c, w, T, vector)
    fitness = fitness_function(c, w, T, vector)
    soluciones.append((vector, total_value, total_weight, fitness))

print("Reporte de soluciones:")
for i, (vector, total_value, total_weight, fitness) in enumerate(soluciones):
    print(f"Vector {i+1}: {vector}, Valor Total: {total_value}, Peso Total: {total_weight}, Fitness: {fitness}")

ruleta_indices = [ruleta(np.array([sol[3] for sol in soluciones])) for _ in range(num_vectores)]
print("Índices seleccionados por ruleta:", ruleta_indices)


Reporte de soluciones:
Vector 1: [1 0 1 0 1 1 1 0 1 0 1 0 1 1 1 0 1 1 0 1], Valor Total: 18232, Peso Total: 1059, Fitness: -7668
Vector 2: [1 0 0 1 0 0 0 0 0 0 1 1 0 0 1 1 0 1 1 1], Valor Total: 12992, Peso Total: 678, Fitness: 12992
Vector 3: [0 1 1 1 1 0 1 1 1 0 0 1 0 0 1 0 0 1 0 1], Valor Total: 16155, Peso Total: 829, Fitness: 13255
Vector 4: [0 0 0 0 0 1 1 0 1 0 0 0 1 1 0 1 1 0 1 1], Valor Total: 12817, Peso Total: 799, Fitness: 12817
Vector 5: [0 0 1 0 0 0 1 0 0 0 0 1 0 1 1 0 1 1 1 0], Valor Total: 11746, Peso Total: 669, Fitness: 11746
Vector 6: [1 0 1 0 1 0 1 0 1 1 1 0 0 0 1 1 0 1 0 1], Valor Total: 15677, Peso Total: 864, Fitness: 9277
Vector 7: [1 0 1 0 1 0 0 1 1 0 1 1 0 1 0 0 0 1 1 1], Valor Total: 15539, Peso Total: 888, Fitness: 6739
Vector 8: [1 1 0 1 0 1 1 1 1 0 1 1 0 1 1 0 0 0 0 1], Valor Total: 17288, Peso Total: 895, Fitness: 7788
Vector 9: [1 0 0 0 0 1 0 1 0 0 1 1 1 1 0 1 0 0 0 1], Valor Total: 11991, Peso Total: 761, Fitness: 11991
Vector 10: [1 0 1 0 0 1 1 1 0 0 0 